# 🏥 MedVision AI (RADMAX-AI) - 1-Click Google Colab Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thamaraiselvang2025aids-oss/RADMAX-AI/blob/main/RADMAX_AI_Colab.ipynb)

This notebook automatically sets up and deploys the **MedVision AI Diagnostics Platform** on Google Colab with GPU acceleration and provides a **public live URL** via Cloudflare Tunnel.

In [ ]:
# @title 1. Clone Repository & Set Up Working Directory
import os

if not os.path.exists("RADMAX-AI"):
    !git clone https://github.com/thamaraiselvang2025aids-oss/RADMAX-AI.git
%cd RADMAX-AI
!git pull
print("✅ Repository ready in:", os.getcwd())

In [ ]:
# @title 2. Install Dependencies & Cloudflared Tunnel Binary
!pip install -q -r requirements.txt
!pip install -q pyngrok uvicorn fastapi pydantic python-multipart

# Download cloudflared for Linux
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

print("✅ All dependencies installed & cloudflared configured!")

In [ ]:
# @title 3. Start MedVision AI & Generate Live Public URL
import subprocess
import time
import re
import urllib.request

# Start FastAPI backend in the background
print("Starting MedVision AI Backend on port 8001...")
backend_process = subprocess.Popen(["python", "ai_service/main.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Allow 6 seconds for models and database to initialize
time.sleep(6)

try:
    res = urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=5)
    print("✅ Backend health status: ONLINE")
except Exception as e:
    print("⚠️ Health check ping:", e)

# Launch Cloudflare quick tunnel to generate public URL
print("\nStarting Cloudflare Tunnel to expose the platform globally...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Listen for and print the public trycloudflare link
public_url = None
for line in tunnel_process.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print("\n" + "="*65)
        print("🎉 MEDVISION AI IS RUNNING & PUBLICLY ACCESSIBLE:")
        print(f"🔗 Web App:   {public_url}")
        print(f"📑 API Docs:  {public_url}/docs")
        print("="*65 + "\n")
        break

# Keep the tunnel process running in foreground
try:
    tunnel_process.wait()
except KeyboardInterrupt:
    print("Stopping services...")
    tunnel_process.terminate()
    backend_process.terminate()

### 💡 Alternative: Localtunnel
If Cloudflare tunnel is ever unavailable in your region, run the cell below to use Localtunnel instead:

In [ ]:
# @title (Alternative) Localtunnel
!npm install -g localtunnel
import urllib.request
password = urllib.request.urlopen('https://loca.lt/mytunnelpassword').read().decode('utf8').strip()
print(f"Your Localtunnel Password (IP) is: {password}")
!npx localtunnel --port 8001